<a href="https://colab.research.google.com/github/Lucaaa31/Anomaly-Segmentation/blob/master/notebooks/Step8_with_Temperature_v2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>


# Step 8 Anomaly Segmentation
---


# Settings


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

%cd /content/drive/MyDrive/Anomaly-Segmentation
#!git pull origin master

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
/content/drive/.shortcut-targets-by-id/1Pz7ReDC4oIzyB7KLXs9SnUMvmnDqYbyB/Anomaly-Segmentation


## Dependencies

In [ ]:
!pip install --upgrade-strategy only-if-needed -r requirements.txt

Traceback (most recent call last):
  File "/usr/local/lib/python3.12/dist-packages/pip/_internal/cli/base_command.py", line 179, in exc_logging_wrapper
    status = run_func(*args)
             ^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/pip/_internal/cli/req_command.py", line 67, in wrapper
    return func(self, options, args)
           ^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/pip/_internal/commands/install.py", line 447, in run
    conflicts = self._determine_conflicts(to_install)
                ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/pip/_internal/commands/install.py", line 578, in _determine_conflicts
    return check_install_conflicts(to_install)
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/pip/_internal/operations/check.py", line 101, in check_install_conflicts
    package_set, _ = create_package_set_from_installed()
              

KeyboardInterrupt: 

## Imports and random seeds

In [ ]:
import os
import glob
import time
import json
import yaml
import random
import warnings
import importlib
from torch.amp.autocast_mode import autocast
import numpy as np
import torch
import torch.nn.functional as F
from PIL import Image
from torchvision.transforms import Compose, Resize, ToTensor
from lightning import seed_everything
from huggingface_hub import hf_hub_download
from huggingface_hub.utils import RepositoryNotFoundError
from ood_metrics import fpr_at_95_tpr
from sklearn.metrics import average_precision_score
import matplotlib.pyplot as plt
import torchvision.transforms.functional as F_tf
import torchvision.transforms as T
from torch.utils.data import TensorDataset, DataLoader


# Utils
from utils.build import build_model_and_data
from utils.class_remap import remap_coco_logits_to_cs


seed_everything(42, verbose=False)
random.seed(42)
np.random.seed(42)
torch.manual_seed(42)
torch.backends.cudnn.deterministic = True
# benchmark must be False together with deterministic=True; for inference the
# speed loss is negligible and runs become reproducible.
torch.backends.cudnn.benchmark = False

---
# Evaluation
## Configuration


In [ ]:
# ===== Configuration =====
# The ONLY knob to change between runs is `model_type`. Run the notebook once per
# model. The table builder uses its own dataset/method/temperature lists, so the
# old per-run dropdowns are gone.
project_root = "/content/drive/MyDrive/Anomaly-Segmentation"
model_type   = "cityscapes"  # @param ["coco", "cityscapes", "phased", "full"]

_CONFIGS_ROOT = project_root + "/configs/dinov2"
if model_type == "coco":
    _cfg      = _CONFIGS_ROOT + "/coco/panoptic/eomt_base_640_2x.yaml"
    _ckpt     = project_root  + "/models/eomt_coco.bin"
    _override = None
elif model_type == "cityscapes":
    _cfg      = _CONFIGS_ROOT + "/cityscapes/semantic/eomt_base_640.yaml"
    _ckpt     = project_root  + "/models/eomt_cityscapes.bin"
    _override = None
else:
    _cfg      = _CONFIGS_ROOT + "/cityscapes/semantic/eomt_base_640.yaml"
    _ckpt     = project_root  + f"/models/coco_finetune/{model_type}/eomt_finetuned_{model_type}.bin"
    _override = {"img_size": (640, 640)}

# Only used to initialise the model builder (setup_data=False); any valid anomaly
# folder works — the table builder builds its own per-dataset paths.
data_path = project_root + "/dataset/Anomaly_Validation_Dataset/RoadAnomaly21"

# Temperature grid swept to pick the per-dataset "best" T (cityscapes run only).
T_GRID = [0.5, 0.75, 0.9, 1.0, 1.1, 1.25, 1.5, 2.0, 2.5]

device = "cuda" if torch.cuda.is_available() else "cpu"

class Args:
    def __init__(self):
        self.model = model_type

args = Args()
print(f"model={model_type}  device={device}")
print(f"ckpt:   {_ckpt}")
print(f"config: {_cfg}")

We build the model and read its metadata (`num_classes`, `img_size`). Temperature is
applied directly to the logits inside `anomaly_score_from_outputs`, so no wrapper is needed.

In [ ]:
print(f"Building the eomt-{model_type}")
model, meta = build_model_and_data(_cfg, _ckpt, data_path, device, setup_data=False, data_overrides=_override)
print(f"  num_classes={meta.num_classes}  img_size={meta.img_size}")
NUM_CLASSES = meta.num_classes
IMG_SIZE = meta.img_size

## Utils methods


In [ ]:
def infer_panoptic(img, target):
    with torch.no_grad(), autocast(dtype=torch.float16, device_type="cuda"):
        imgs = [img.to(device)]
        img_sizes = [img.shape[-2:] for img in imgs]

        transformed_imgs = model.resize_and_pad_imgs_instance_panoptic(imgs)
        mask_logits_per_layer, class_logits_per_layer = model(transformed_imgs)

        mask_logits = F.interpolate(
            mask_logits_per_layer[-1], model.img_size, mode="bilinear"
        )
        mask_logits = model.revert_resize_and_pad_logits_instance_panoptic(
            mask_logits, img_sizes
        )

        preds = model.to_per_pixel_preds_panoptic(
            mask_logits,
            class_logits_per_layer[-1],
            model.stuff_classes,
            model.mask_thresh,
            model.overlap_thresh,
        )[0].cpu()

    pred = preds.numpy()
    sem_pred, inst_pred = pred[..., 0], pred[..., 1]

    target_seg = model.to_per_pixel_targets_panoptic([target])[0].cpu().numpy()
    sem_target, inst_target = target_seg[..., 0], target_seg[..., 1]

    cls_logits = class_logits_per_layer[-1][0][:, :NUM_CLASSES].float()
    masks_probs = torch.sigmoid(mask_logits[0]).float()

    num_q = masks_probs.shape[0]
    H, W = masks_probs.shape[1], masks_probs.shape[2]

    semantic_logits = torch.mm(
        cls_logits.t(),
        masks_probs.view(num_q, -1),
    ).view(NUM_CLASSES, H, W)

    return sem_pred, inst_pred, sem_target, inst_target, semantic_logits, cls_logits, masks_probs

In [ ]:

def draw_black_border(sem, inst, mapping):
    h, w = sem.shape
    out = np.zeros((h, w, 3))
    for s in np.unique(sem):
        out[sem == s] = mapping[s]

    combined = sem.astype(np.int64) * 100000 + inst.astype(np.int64)
    border = np.zeros((h, w), dtype=bool)
    border[1:, :] |= combined[1:, :] != combined[:-1, :]
    border[:-1, :] |= combined[1:, :] != combined[:-1, :]
    border[:, 1:] |= combined[:, 1:] != combined[:, :-1]
    border[:, :-1] |= combined[:, 1:] != combined[:, :-1]
    out[border] = 0
    return out


def plot_panoptic_results(img, sem_pred, inst_pred, sem_target, inst_target):
    all_ids = np.union1d(np.unique(sem_pred), np.unique(sem_target))
    mapping = {
        s: (
            [0, 0, 0]
            if s == -1 or s == model.num_classes
            else plt.cm.hsv(i / len(all_ids))[:3]
        )
        for i, s in enumerate(all_ids)
    }

    vis_pred = draw_black_border(sem_pred, inst_pred, mapping)
    vis_target = draw_black_border(sem_target, inst_target, mapping)

    img_np = (
        img.cpu().numpy().transpose(1, 2, 0) if img.dim() == 3 else img.cpu().numpy()
    )

    fig, axes = plt.subplots(1, 3, figsize=(15, 5))
    axes[0].imshow(img_np)
    axes[0].set_title("Input")
    axes[1].imshow(vis_pred)
    axes[1].set_title("Prediction")
    axes[2].imshow(vis_target)
    axes[2].set_title("Target")

    for ax in axes:
        ax.axis("off")

    plt.tight_layout()
    plt.show()

## Post hoc methods

In [ ]:
def get_msp(logits):

    probs = F.softmax(logits.float(), dim=0)
    msp = probs.max(dim=0).values              # [H, W]
    return (1.0 - msp).detach().cpu().numpy().astype("float32")


def get_maxlogit(logits):
    max_logit = logits.float().max(dim=0).values
    return (-max_logit).detach().cpu().numpy().astype("float32")


def get_entropy(logits):

    logits = logits.float()
    probs = F.softmax(logits, dim=0)
    num_classes = logits.shape[0]
    entropy = -(probs * (probs + 1e-7).log()).sum(dim=0)   # [H, W]
    norm_entropy = entropy / torch.log(
        torch.tensor(num_classes, dtype=torch.float32, device=logits.device)
    )
    return norm_entropy.detach().cpu().numpy().astype("float32")


def get_Rba(cls_logits, masks_probs):


    cls_logits  = cls_logits.float()
    masks_probs = masks_probs.float()

    query_class_score = F.softmax(cls_logits, dim=1).max(dim=1).values

    rba_score = (query_class_score[:, None, None] * masks_probs).max(dim=0).values

    return (1.0 - rba_score).detach().cpu().numpy().astype("float32")

## Anomaly score for one image (given a temperature)
This helper applies a temperature `T` to the model outputs and returns the per-pixel anomaly score for the selected post-hoc method. It is used both for normal inference and for the `best`-temperature search, so the scoring logic lives in a single place.

In [ ]:
def anomaly_score_from_outputs(result_logits, cls_logits_raw, masks_probs_raw, T, method):
    """Compute the anomaly score map for one image at temperature T.

    result_logits   : semantic logits  [C, H, W]   (already remapped if COCO)
    cls_logits_raw  : query class logits [Q, C(+1)]
    masks_probs_raw : query mask probs   [Q, H, W]
    T               : scalar temperature (float or 0-dim tensor)
    method          : one of "MSP", "max_logit", "max_entropy", "RbA"
    """
    scaled_logits = result_logits / T
    match method:
        case "MSP":
            return get_msp(scaled_logits)
        case "max_logit":
            return get_maxlogit(scaled_logits)
        case "max_entropy":
            return get_entropy(scaled_logits)
        case "RbA":
            # for RbA temperature is applied to the query class logits
            return get_Rba(cls_logits_raw / T, masks_probs_raw)
    raise ValueError(f"Unknown method: {method}")

## Preprocessing

In [ ]:
def input_transform(img):
    img_resized = F_tf.resize(img, IMG_SIZE, interpolation=T.InterpolationMode.BILINEAR)
    img_tensor = F_tf.to_tensor(img_resized)
    return (img_tensor * 255).to(torch.uint8).to(device)

def target_transform(img):
    img_resized = F_tf.resize(img, IMG_SIZE, interpolation=T.InterpolationMode.NEAREST)
    return np.array(img_resized)

## Anomaly GT remap helper
Same dataset-specific OoD label remapping you already had, factored out so it can be reused by the collection pass and the temperature search.

In [ ]:
# Dataset-specific OoD-label remap and GT-path resolution. Both match on the
# EXACT dataset name (passed in) instead of `in pathGT` — the substring test made
# "RoadAnomaly" also fire for "RoadAnomaly21". Convention after remap:
# 1 = anomaly, 0 = in-distribution, anything else (e.g. 255) = ignore.
# Mask encodings were verified per dataset:
#   RoadAnomaly21 / RoadObsticle21 / fs_static / FS_LostFound_full -> 0/1/255
#   RoadAnomaly -> 0 / 2  (2 = anomaly, remapped to 1)
def remap_ood_gt(target_np, dataset):
    ood_gts = target_np.copy()
    if dataset == "RoadAnomaly":
        ood_gts = np.where((ood_gts == 2), 1, ood_gts)
    # all other datasets already use 0 / 1 / 255 -> no remap
    return ood_gts


def gt_path_for(path, dataset):
    pathGT = path.replace("images", "labels_masks")
    if dataset == "RoadObsticle21":
        pathGT = pathGT.replace("webp", "png")
    elif dataset == "fs_static":
        pathGT = pathGT.replace("jpg", "png")
    elif dataset == "RoadAnomaly":
        pathGT = pathGT.replace("jpg", "png")
    return pathGT

---
# Build the Step-8 tables in one run

For the **selected `model_type`** (Configuration cell), this section caches each dataset once
and derives **all four post-hoc methods** (MSP / MaxLogit / MaxEntropy / RbA) on the **five
anomaly benchmarks**. It produces two separate outputs:

1. **Anomaly detection** (every `model_type`): methods × datasets at baseline **T = 1.0**.
   → `results/step8_anomaly_{model}.txt`
2. **Temperature scaling** (**only `model_type == "cityscapes"`**): per method, a sweep over
   `TABLE_TEMPS` (`T=1.0 / 0.5 / 0.75 / 1.1 / best`), where `best` is the `T_GRID` value
   maximising AuPRC on each dataset.
   → `results/step8_temperature_cityscapes.txt`

Properties (parity with the corrected Step 7):

- **One forward per image** — single execution covers all methods.
- **Exact dataset matching** in the GT remap (no `"RoadAnomaly"` firing for `RoadAnomaly21`).
- **Per-dataset checkpoint** in `checkpoints/step8/step8_checkpoint_{model}.json` — re-running
  skips finished datasets, surviving Colab disconnects.
- **4.5 h time budget** — stops gracefully before the GPU session is killed.

Run once per `model_type` to fill the four anomaly tables; temperature scaling is computed only
on the `cityscapes` run.

The dropdowns (`dataset_to_use`, `methods`, `temperature`) only drive the earlier single-image
demo cells; the builder ignores them and uses `TABLE_DATASETS` / `METHODS_TABLE` / `TABLE_TEMPS`.

**Note:** mIoU is not recomputed (it depends only on the weights). Fill it from Step 4/5.


In [ ]:
def cache_dataset(ds_name):
    """Run the model once over one anomaly dataset and cache per-image outputs.
    Returns a list of dicts (same structure as `cached_outputs`).
    All four post-hoc methods are later derived from this single cache."""
    ds_data_path = project_root + f"/dataset/Anomaly_Validation_Dataset/{ds_name}"
    ds_input_pattern = ds_data_path + "/images/*"

    out = []
    for path in sorted(glob.glob(os.path.expanduser(str(ds_input_pattern)))):
        raw_image = Image.open(path).convert('RGB')
        images = input_transform(raw_image)

        pathGT = gt_path_for(path, ds_name)
        raw_target = Image.open(pathGT).convert('L')
        target_np = target_transform(raw_target)

        unique_ids = np.unique(target_np)
        unique_ids = unique_ids[unique_ids != 0]

        masks, labels = [], []
        for s_id in unique_ids:
            masks.append(target_np == s_id)
            labels.append(s_id)

        if len(masks) > 0:
            target_dict = {
                "masks": torch.from_numpy(np.stack(masks)).bool().to(device),
                "labels": torch.from_numpy(np.array(labels)).long().to(device),
            }
        else:
            target_dict = {
                "masks": torch.zeros((0, target_np.shape[0], target_np.shape[1]), dtype=torch.bool).to(device),
                "labels": torch.zeros((0,), dtype=torch.long).to(device),
            }

        (sem_pred, inst_pred, sem_target, inst_target,
         result_logits, cls_logits_raw, masks_probs_raw) = infer_panoptic(images, target_dict)

        if args.model == "coco":
            result_logits = remap_coco_logits_to_cs(result_logits)

        ood_gts = remap_ood_gt(target_np, ds_name)
        if 1 not in np.unique(ood_gts):
            del result_logits, cls_logits_raw, masks_probs_raw
            torch.cuda.empty_cache()
            continue

        out.append({
            "result_logits":   result_logits.detach().cpu(),
            "cls_logits_raw":  cls_logits_raw.detach().cpu(),
            "masks_probs_raw": masks_probs_raw.detach().cpu(),
            "ood_gts":         ood_gts,
        })
        del result_logits, cls_logits_raw, masks_probs_raw
        torch.cuda.empty_cache()

    return out

In [ ]:
def evaluate_cached(cache, T, method):
    """Same as evaluate_at_temperature but on an explicit cache + method."""
    anomaly_score_list, ood_gts_list = [], []
    for item in cache:
        result_logits   = item["result_logits"].to(device)
        cls_logits_raw  = item["cls_logits_raw"].to(device)
        masks_probs_raw = item["masks_probs_raw"].to(device)

        anomaly_result = anomaly_score_from_outputs(
            result_logits, cls_logits_raw, masks_probs_raw, T, method
        )
        ood_gts_list.append(item["ood_gts"])
        anomaly_score_list.append(anomaly_result)

        del result_logits, cls_logits_raw, masks_probs_raw
        torch.cuda.empty_cache()

    ood_gts = np.array(ood_gts_list)
    anomaly_scores = np.array(anomaly_score_list)

    ood_out = anomaly_scores[ood_gts == 1]
    ind_out = anomaly_scores[ood_gts == 0]

    val_out = np.concatenate((ind_out, ood_out))
    val_label = np.concatenate((np.zeros(len(ind_out)), np.ones(len(ood_out))))

    prc_auc = average_precision_score(val_label, val_out)
    fpr = fpr_at_95_tpr(val_out, val_label)
    return prc_auc, fpr

In [ ]:
# ============================================================================
# Single-run Step-8 builder.
#   * each dataset is cached ONCE (one model forward per image)
#   * all 4 methods are derived from that one cache (free)
#   * ANOMALY DETECTION (baseline T=1.0) is computed for every model_type
#   * TEMPERATURE SCALING is computed ONLY for model_type == "cityscapes"
#   * per-dataset checkpoint -> resume after a Colab disconnect
#   * 4.5 h time budget      -> stop gracefully, finish on the next run
# ============================================================================
METHODS_TABLE  = ["MSP", "max_logit", "max_entropy", "RbA"]
TABLE_DATASETS = ["RoadAnomaly21", "RoadObsticle21", "FS_LostFound_full", "fs_static", "RoadAnomaly"]
TABLE_TEMPS    = ["none", 0.5, 0.75, 1.1, "best"]   # "none" == baseline T = 1.0

COL_LABELS = {
    "RoadAnomaly21":     "SMIYC RA-21",
    "RoadObsticle21":    "SMIYC RO-21",
    "FS_LostFound_full": "FS L&F",
    "fs_static":         "FS Static",
    "RoadAnomaly":       "Road Anomaly",
}

# Temperature scaling only makes sense / is requested for the Cityscapes model.
DO_TEMPERATURE = (args.model == "cityscapes")

CKPT_DIR  = project_root + "/checkpoints/step8"
os.makedirs(CKPT_DIR, exist_ok=True)
os.makedirs("results", exist_ok=True)
CKPT_PATH    = f"{CKPT_DIR}/step8_checkpoint_{args.model}.json"
TIME_LIMIT_S = 4.5 * 3600

def _tkey(t):
    return str(t)

# --- resume from checkpoint -------------------------------------------------
# results[method][temp_key][dataset] = [auprc%, fpr95%, T_used]
results = {}
if os.path.exists(CKPT_PATH):
    with open(CKPT_PATH) as f:
        results = json.load(f)
    print("Resumed checkpoint. Completed datasets:", results.get("_done", []))
results.setdefault("_done", [])

print(f"model={args.model} | temperature scaling: {'ON' if DO_TEMPERATURE else 'OFF (baseline only)'}")

t0 = time.time()
out_of_time = False

for ds in TABLE_DATASETS:
    if ds in results["_done"]:
        print(f"[skip] {ds} already in checkpoint")
        continue
    if time.time() - t0 > TIME_LIMIT_S:
        print(f"[time] budget reached before {ds}; stopping (resume next run)")
        out_of_time = True
        break

    print(f"\n=== Caching {ds} ===")
    cache = cache_dataset(ds)
    print(f"  cached {len(cache)} OoD images")

    for method in METHODS_TABLE:
        # baseline (T = 1.0) — always
        a, f = evaluate_cached(cache, 1.0, method)
        (results.setdefault(method, {}).setdefault("none", {})[ds]) = [a * 100.0, f * 100.0, 1.0]
        print(f"  {method:12s} baseline AuPRC={a*100:6.2f}%  FPR95={f*100:6.2f}%")

        if DO_TEMPERATURE:
            # per-dataset best T for this method (AuPRC, FPR95 as tie-break)
            best_T, best_a, best_f = None, -1.0, None
            for T in T_GRID:
                aa, ff = evaluate_cached(cache, float(T), method)
                if (aa > best_a) or (aa == best_a and ff < best_f):
                    best_T, best_a, best_f = float(T), aa, ff
            for temp in TABLE_TEMPS:
                if temp == "none":
                    continue  # already computed above
                elif temp == "best":
                    T, aa, ff = best_T, best_a, best_f
                else:
                    T = float(temp)
                    aa, ff = evaluate_cached(cache, T, method)
                (results[method].setdefault(_tkey(temp), {})[ds]) = [aa * 100.0, ff * 100.0, T]
            print(f"               temp sweep done (best T={best_T})")

    results["_done"].append(ds)
    with open(CKPT_PATH, "w") as fj:
        json.dump(results, fj, indent=2)
    print(f"[ckpt] saved -> {CKPT_PATH}")

    del cache
    torch.cuda.empty_cache()

print("\nDone." if not out_of_time else "\nStopped early — re-run this cell to continue.")

In [ ]:
# ============================================================================
# Render the tables from the checkpoint (can run standalone, no recompute).
#   (1) anomaly-detection table  -> 4 methods x 5 datasets, baseline T=1.0
#                                   (same layout as the report image)
#   (2) temperature-scaling tables -> one per method (only for cityscapes)
# Missing cells are shown as "-".
# ============================================================================
with open(CKPT_PATH) as f:
    results = json.load(f)

MODEL_NAME = f"EoMT-{args.model}"
ROWS = [("MSP", "MSP"), ("MaxLogit", "max_logit"),
        ("MaxEntropy", "max_entropy"), ("RbA", "RbA")]
COLS = [("SMIYC RA-21",  "RoadAnomaly21"),
        ("SMIYC RO-21",  "RoadObsticle21"),
        ("FS L&F",       "FS_LostFound_full"),
        ("FS Static",    "fs_static"),
        ("Road Anomaly", "RoadAnomaly")]
W_MODEL, W_MIOU, W_METHOD, W_CELL = 12, 6, 12, 9

def _cell(method_key, ds_key, temp_key="none"):
    try:
        a, f, _T = results[method_key][temp_key][ds_key]
        return (f"{a:.2f}", f"{f:.2f}")
    except (KeyError, TypeError):
        return ("-", "-")

def render_anomaly():
    """Anomaly-detection table: methods x datasets at baseline T = 1.0."""
    line1 = " " * (W_MODEL + 1 + W_MIOU + 1 + W_METHOD)
    for disp, _ in COLS:
        line1 += " " + disp.center(2 * W_CELL + 1)
    line2 = f"{'Model':<{W_MODEL}} {'mIoU':<{W_MIOU}} {'Method':<{W_METHOD}}"
    for _ in COLS:
        line2 += f" {'AuPRC':>{W_CELL}} {'FPR95':>{W_CELL}}"
    sep = "-" * len(line2)
    body = []
    for i, (rdisp, rkey) in enumerate(ROWS):
        model = MODEL_NAME if i == 0 else ""
        miou  = "----"     if i == 0 else ""
        row = f"{model:<{W_MODEL}} {miou:<{W_MIOU}} {rdisp:<{W_METHOD}}"
        for _, dkey in COLS:
            a, fp = _cell(rkey, dkey, "none")
            row += f" {a:>{W_CELL}} {fp:>{W_CELL}}"
        body.append(row)
    title = f"Step 8 — EoMT anomaly detection (corrected) — model={MODEL_NAME}, T=1.0"
    return "\n".join([title, "", line1, line2, sep, *body, ""])

def render_temp_sweep(method_key, method_disp):
    """Temperature-scaling table for one method: temperatures x datasets."""
    temps = ["none", "0.5", "0.75", "1.1", "best"]
    tlabel = {"none": "T=1.0", "0.5": "T=0.5", "0.75": "T=0.75",
              "1.1": "T=1.1", "best": "best T"}
    line1 = " " * W_METHOD
    for disp, _ in COLS:
        line1 += " " + disp.center(2 * W_CELL + 1)
    line2 = f"{'Temp':<{W_METHOD}}"
    for _ in COLS:
        line2 += f" {'AuPRC':>{W_CELL}} {'FPR95':>{W_CELL}}"
    sep = "-" * len(line2)
    body = []
    for t in temps:
        row = f"{tlabel[t]:<{W_METHOD}}"
        for _, dkey in COLS:
            a, fp = _cell(method_key, dkey, t)
            row += f" {a:>{W_CELL}} {fp:>{W_CELL}}"
        body.append(row)
    return "\n".join([f"[{method_disp}] temperature sweep", line1, line2, sep, *body, ""])

DO_TEMPERATURE = (args.model == "cityscapes")

print(render_anomaly())
if DO_TEMPERATURE:
    print(f"\n=== Temperature scaling ({MODEL_NAME}) ===\n")
    for disp, key in ROWS:
        print(render_temp_sweep(key, disp))
else:
    print(f"\n(temperature scaling not computed for model={args.model})")

In [ ]:
# ---- Save results to SEPARATE .txt files: anomaly detection vs temperature ----
os.makedirs("results", exist_ok=True)

# (1) anomaly detection (every model)
ANO_TXT = f"results/step8_anomaly_{args.model}.txt"
with open(ANO_TXT, "w") as f:
    f.write(render_anomaly())
print("Saved ->", ANO_TXT)

# (2) temperature scaling (cityscapes only)
if DO_TEMPERATURE:
    TEMP_TXT = f"results/step8_temperature_{args.model}.txt"
    sections = [f"Step 8 — EoMT temperature scaling (corrected) — model={MODEL_NAME}", ""]
    sections += [render_temp_sweep(k, d) for d, k in ROWS]
    with open(TEMP_TXT, "w") as f:
        f.write("\n".join(sections))
    print("Saved ->", TEMP_TXT)
else:
    print(f"(no temperature file for model={args.model})")